# 13 · Does the criterion follow the distribution, and how fast

**What this shows.** Chapter 3's Fig 3.6, for both Hard designs: the daily alternation of the opto mice (masking
days; laser days, on laser-off trials, as a check) and the blocks of days of the pre-opto mice. After a switch the
criterion moves from where it was (0) towards the normative criterion of the new distribution (1).

**One number per mouse, by design.**
- *Days*: `convergence_100_300`, the criterion averaged over trials 100–300 of the day (inside the first day, so the
  slow drift across days hardly enters), a mouse's five or six days of one kind combined.
- *Blocks*: `block_convergence`, the whole block's PSE against the previous phase's (4-parameter fits on thousands
  of trials) as a share of the normative shift: 1 = moved by the normative amount, 0 = did not move. A mouse has one
  block switch of each kind, the distance to go can be 0.01–0.06 and a 50-trial window moves by 0.3–0.6, so the
  curve's numbers do not hold for one block, so the figure shows the blocks by this number alone (the curve, a
  description, is on the switch report's blocked page). Hard-A → Hard-B is the check: the slow upward drift of the
  criterion across days works against it.

**How the curve is measured** (`behaviour/adaptation.py`). The criterion is the 50 % point of the psychometric with
slope and lapses fixed at the fit of its phase (the block, or every day of one distribution and session type pooled;
the Uniform phase takes the expert-Uniform fit), in running 50-trial windows; a fit that ends at ±1 (the mouse chose
mostly one side) is missing. One shape for every phase would read a change of lapses between phases as a criterion
shift. 0 = the criterion over the last 250 trials before the switch, 1 = the normative criterion at the mouse's
expert-Uniform sigma; not clipped. A mouse's switches of one kind are combined weighted by |normative − before|,
then the mice are averaged with a 95 % t interval. Mice the bias check flags (`bias.csv`, the same list in every
report) are left out.

**What produced it.** Two `switch_adaptation` runs:

```bash
sc-reports switch-adaptation --cohort behaviour1-cohort
sc-reports switch-adaptation --cohort opto1-cohort
```

In [ ]:
from nb_setup import *
from sound_categorisation.reports.switches import DAY_KINDS, DIRECTIONS, block_table, pooled_curve, t_interval

banner()
runs = {c: open_run('switch_adaptation', c) for c in dict.fromkeys([SWITCH_COHORT, OPTO_COHORT])}
tables = [load_switches(r) for r in runs.values()]


def read(name):
    frames = [t[name] for t in tables if name in t and len(t[name])]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def clean(df):
    return df[~df['animal'].isin(biased)] if len(df) else df


conv, animals, switches, bias, overnight, pre_post = (
    read(n) for n in ('convergence', 'animals', 'switches', 'bias', 'overnight', 'pre_post'))
biased = sorted(set(bias.loc[bias['biased_animal'].astype(bool), 'animal']))
LABELS = ['animal', 'switch_idx', 'from_distribution', 'to_distribution', 'transition', 'design', 'switch_kind']
wide = switches.groupby(LABELS + ['stat'])['value'].first().unstack('stat').reset_index()
blocks = clean(block_table(pre_post))
print('left out by the bias check:', biased or 'none')
animals.groupby(['design', 'switch_kind']).size()

## 1 · One mouse through the pipeline

`compute_switches` finds the mouse's switches (`hard_design` decides whether they are blocks or days) and measures
each one: the criterion before the switch (with the previous phase's shape), the curve after it (with the new
phase's shape) and where the block or day ends. The shapes are in `pre_shape_*` and `shape_*`.

In [ ]:
from sound_categorisation.behaviour.adaptation import compute_switches, hard_design

exp = load_experiment()
candidates = sorted(set(animals.loc[animals['design'] == 'blocked', 'animal']) - set(biased)) or sorted(set(animals['animal']))
aid = candidates[0]
kw = dict(min_block_trials=300, max_trials=800) if SYNTHETIC else {}
res = compute_switches(exp.animals[aid], **kw)
print(aid, hard_design(exp.animals[aid]), *res, sep='\n')
r = res[min(1, len(res) - 1)]
print('shape before:', r.pre[['shape_sigma', 'shape_lapse_low', 'shape_lapse_high']].round(3).to_dict())
print('shape after: ', r.dynamics[['shape_sigma', 'shape_lapse_low', 'shape_lapse_high']].round(3).to_dict())
r.convergence[r.convergence['method'] == 'pinned_running'].head(8).round(3)

## 2 · The figure

Left: the criterion after a day's switch, each kind of day averaged over the clean mice, with its 95 % t interval;
the grey band marks trials 100–300. The blocked curve is left out: with one switch of each kind per mouse it swings
between −1 and 2 and would hide the days (it is on the switch report's blocked page, as a description). Middle: the days, one number per mouse
(the mean over trials 100–300), with the mean over mice and its 95 % t interval. Right: the blocks, one number per
mouse and direction (the whole block's share of the normative shift). The slow upward drift across days works
against Hard-A → Hard-B, so a mouse above 0 there moved with the distribution despite it.

In [ ]:
day_kinds = [('daily', k) for k in DAY_KINDS]
colours = {'masking day': PALETTE[2], 'laser day': PALETTE[3]}
direction_colours = dict(zip(DIRECTIONS, ('0.55', PALETTE[0], PALETTE[1])))


def dots(ax, i, values, colour):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    ax.plot(i + (np.linspace(-0.15, 0.15, len(v)) if len(v) > 1 else np.zeros(len(v))), v, 'o', color=colour, alpha=0.75)
    m, lo, hi, n = t_interval(v)
    if np.isfinite(m):
        ax.plot([i - 0.25, i + 0.25], [m, m], color='k', lw=1.6)
    if np.isfinite(lo):
        ax.plot([i, i], [lo, hi], color='k', lw=1.2)
    return n


fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.4, 0.7, 1]})
ax = axes[0]
cv = clean(conv[conv['method'] == 'pinned_running'])
for design, kind in day_kinds:
    per = pooled_curve(cv[(cv['design'] == design) & (cv['switch_kind'] == kind)])
    if not len(per):
        continue
    g = pd.DataFrame([(t, *t_interval(d['convergence'])) for t, d in per.groupby('trial')],
                     columns=['trial', 'mean', 'lo', 'hi', 'n'])
    ax.plot(g['trial'], g['mean'], color=colours[kind], lw=2, label=f'{kind} (n = {int(g["n"].max())} mice)')
    ax.fill_between(g['trial'], g['lo'], g['hi'], color=colours[kind], alpha=0.15, lw=0)
ax.axvspan(100, 300, color='0.93', zorder=0)
ax.set(xscale='log', xlim=(20, 600), ylim=(-1, 2), xlabel='trials after the switch (window centre, log)',
       ylabel='days: convergence (0 = before, 1 = normative)')
ax.set_xticks([25, 50, 100, 200, 400])
ax.set_xticklabels(['25', '50', '100', '200', '400'])
ax.minorticks_off()
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
an = clean(animals)
for i, (design, kind) in enumerate(day_kinds):
    dots(ax, i, an.loc[(an['design'] == design) & (an['switch_kind'] == kind), 'convergence_100_300'], colours[kind])
ax.set_xticks(range(len(day_kinds)))
ax.set_xticklabels([k for _, k in day_kinds])
ax.set(xlim=(-0.6, len(day_kinds) - 0.4), ylim=(-1, 2), ylabel='days: convergence over trials 100-300')

ax = axes[2]
directions = [d for d in DIRECTIONS if d in set(blocks['direction'])]
for i, d in enumerate(directions):
    dots(ax, i, blocks.loc[blocks['direction'] == d, 'block_convergence'], direction_colours[d])
ax.set_xticks(range(len(directions)))
ax.set_xticklabels([d.replace(' → ', ' →\n') + ('\n(drift works against)' if d == 'Hard-A → Hard-B' else '')
                    for d in directions], fontsize=8)
ax.set(xlim=(-0.6, max(len(directions), 1) - 0.4), ylabel='blocks: share of the normative shift (whole block)')
for ax in axes:
    ax.axhline(1, ls='--', color='k', lw=0.9)
    ax.axhline(0, color='0.6', lw=0.8)
plt.tight_layout()

## 3 · The numbers

Days: per kind of switch, the clean mice's `convergence_100_300` with its 95 % t interval. Blocks: per direction,
their `block_convergence`. The unit is the mouse; a mouse's switches of one kind are combined first. Below, every
mouse.

In [ ]:
rows = []
for kind, g in an[an['design'] == 'daily'].groupby('switch_kind'):
    m, lo, hi, n = t_interval(g['convergence_100_300'])
    rows.append({'design': 'daily', 'switch': kind, 'measure': 'convergence_100_300', 'mice': n, 'mean': m, 'low': lo, 'high': hi})
for d in directions:
    m, lo, hi, n = t_interval(blocks.loc[blocks['direction'] == d, 'block_convergence'])
    rows.append({'design': 'blocked', 'switch': d, 'measure': 'block_convergence', 'mice': n, 'mean': m, 'low': lo, 'high': hi})
display(pd.DataFrame(rows).round(2))
display(an[an['design'] == 'daily'].pivot_table(index='animal', columns='switch_kind', values='convergence_100_300').round(2))
blocks.pivot_table(index='animal', columns='direction', values='block_convergence').reindex(columns=directions).round(2)

## 4 · The blocks, switch by switch

`pre_post.csv`: the pooled PSE of the previous phase (`pse_pre`; the last five Uniform sessions before the first
switch) and of the whole new block (`pse_post`), both 4-parameter fits, their difference, the normative shift and
the share of it. A block whose PSE lies outside the stimulus range (the mouse chose mostly one side) is not a
measurement and is left empty. Beside it, the curve's end (`convergence_final`, the last 250 trials of the block
against the last 250 before the switch), which divides by the distance still to go and so swings with it.

In [ ]:
pp = clean(pre_post) if len(pre_post) else pre_post
if len(pp):
    pp = pp.assign(switch=pp['from_distribution'] + ' → ' + pp['to_distribution'])
    ends = wide[['animal', 'switch_idx', 'convergence_final']] if 'convergence_final' in wide else wide[['animal', 'switch_idx']]
    pp = pp.merge(ends, on=['animal', 'switch_idx'], how='left')
    display(pp[['animal', 'switch_idx', 'switch', 'pse_pre', 'pse_post', 'delta', 'normative_shift', 'block_convergence',
                'convergence_final']].round(3))
else:
    print('no blocked switches in these runs')

## 5 · Overnight

`overnight.csv` (blocks): the criterion over the last 100 trials of a session against the first 100 of the next.
`toward_pre` signs the change by the direction of the switch: > 0 = back towards the previous distribution
overnight. Signed this way, noise in either window cannot set the sign, which it did when the end-of-session value
itself chose it.

In [ ]:
ov = clean(overnight[overnight['design'] == 'blocked']) if len(overnight) else overnight
if len(ov) and ov['toward_pre'].notna().any():
    per = ov.groupby('animal')['toward_pre'].median()
    m, lo, hi, n = t_interval(per)
    print(f'overnight change back towards the previous distribution, median per mouse: mean {m:+.3f} '
          f'(95 % t interval {lo:+.3f} to {hi:+.3f}, {n} mice)')
    display(ov.groupby('animal').agg(pairs=('toward_pre', 'count'), median_toward_pre=('toward_pre', 'median'),
                                     median_change=('delta', 'median')).round(3))
else:
    print('no overnight pairs in these runs')